# Establishing Risk Factors for CVD - Logistic Model


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    roc_curve,
    roc_auc_score,
)

import matplotlib.pyplot as plt

bangladesh_data = pd.read_csv(
    "/Users/lilahduboff/Documents/Duke_Machine_Learning/Final/IDS705_ML_Final_Project_Cardiovasular_Disease/00_Cleaned_Data/bangladesh_cleaned_data.csv"
)

cleveland_data = pd.read_csv(
    "/Users/lilahduboff/Documents/Duke_Machine_Learning/Final/IDS705_ML_Final_Project_Cardiovasular_Disease/00_Cleaned_Data/uci_cleveland_data.csv"
)

In [4]:
cleveland_data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [5]:
bangladesh_data.head()

,Sex,Age,Weight (kg),Height (m),BMI,Abdominal Circumference (cm),Blood Pressure (mmHg),Total Cholesterol (mg/dL),HDL (mg/dL),Fasting Blood Sugar (mg/dL),...,Physical Activity Level,Family History of CVD,CVD Risk Level,Height (cm),Waist-to-Height Ratio,Systolic BP,Diastolic BP,Blood Pressure Category,Estimated LDL (mg/dL),CVD Risk Score
0,F,32.0,69.1,1.71,23.6,86.2,125/79,248.0,78.0,111.0,...,Low,N,INTERMEDIARY,171.0,0.504,125.0,79.0,Elevated,140.0,17.93
1,F,55.0,118.7,1.69,41.6,82.5,139/70,162.0,50.0,135.0,...,High,Y,HIGH,169.0,0.488,139.0,70.0,Hypertension Stage 1,82.0,20.51
2,M,NaN,NaN,1.83,26.9,106.7,104/77,103.0,73.0,114.0,...,High,Y,INTERMEDIARY,183.0,0.583,104.0,77.0,Normal,0.0,12.64
3,M,44.0,108.3,1.80,33.4,96.6,140/83,134.0,46.0,91.0,...,High,Y,INTERMEDIARY,NaN,0.537,140.0,83.0,Hypertension Stage 1,58.0,16.36
4,F,32.0,99.5,1.86,28.8,102.7,144/83,146.0,64.0,141.0,...,High,N,INTERMEDIARY,186.0,0.552,144.0,83.0,Hypertension Stage 1,52.0,17.88


### Inspect the data a bit more closely 

>So from the exploration, we were trying to figure out what the data looked like, if there were missing values, if any additional information would be needed, etc. Now we can take a closer look and start preprocessing. In particular, I may opt to numericize and one-hot-encode the bangladesh data, as it does have missing values, and the categorical aspect may make it challenging to compelete a logistic regression. Here are some ideas I will try:

>It doesn't make sense to run a logistic regression on data with no binary outcome, so for the bangladesh data, the preprocessing would likely need to include risk/no risk, rather than high, medium, and low. This poses other issues, as the results would not be explicitly accurate to how a real diagnosis may play out. However, it is the closest we could get. Another thing to consider in this regard is the fact that our classes may be imbalanced, which could influence model results. Handling this will need to be very careful.

>The readme and codebook for the UCI cleveland data states that the predicted outcome variable, num, represents the degree to which blood vessels are constricted. 0 if constricted less than 50%, and 1 if constricted over 50%. However, it seems like there are 2's in the mix. I would assume this is a severe category, but additional resources may need to be considered based on other biometric measurements. Ie, if chol and fbs are critical, we could somewhat assume the individual is at some risk, and replace the 2 to a 1. In doing so, we are potentially losing other important information, but would be creating a binary system, makeing modeling and interpretation easier. 

>*Finally, the cleveland data has question mark values that seem unable to be removed. This is weird and I will need to refer to a professor for next steps. Until then, we will treat them as missing. 

In [7]:
cleveland_data["num"].value_counts()

0    164
1     55
2     36
3     35
4     13
Name: num, dtype: int64

bro what the literal hell is this

I guess if we sum 1-4, as being "at risk" the classes are not imbalanced

>ok so we know that less than 100mg/dL is normal level for fbs, 100-125 is prediabetic, and over that is diabetic and/or dangerous. Based on that, we can find out summaries of fbs for each risk category, since we now know it's not binary. However, fbs IS, for 1=fbs over 120mg/dL

In [ ]:
cleveland_data_fbs_grouped = cleveland_data.groupby("num")["fbs"].value_counts()

cleveland_data_fbs_grouped

num  fbs
0    0.0    141
     1.0     23
1    0.0     51
     1.0      4
2    0.0     27
     1.0      9
3    0.0     27
     1.0      8
4    0.0     12
     1.0      1
Name: fbs, dtype: int64

>Ok this makes NO sense, we'd expect the values of 1's to increase with each subsequent risk level.... let's try a different variable

In [13]:
cleveland_data_chol_grouped = cleveland_data.groupby("num")["chol"].mean()

cleveland_data_chol_grouped

num
0    242.640244
1    249.109091
2    259.277778
3    246.457143
4    253.384615
Name: chol, dtype: float64

>ok let's go basic, and do age and bmi

In [ ]:
cleveland_data_age_grouped = cleveland_data.groupby("num")["age"].mean()

cleveland_data_age_grouped

num
0    52.585366
1    55.381818
2    58.027778
3    56.000000
4    59.692308
Name: age, dtype: float64

In [17]:
cleveland_data_angina_grouped = cleveland_data.groupby("num")["exang"].value_counts()

cleveland_data_angina_grouped

num  exang
0    0.0      141
     1.0       23
1    0.0       30
     1.0       25
2    1.0       22
     0.0       14
3    1.0       23
     0.0       12
4    0.0        7
     1.0        6
Name: exang, dtype: int64

In [8]:
bangladesh_data["CVD Risk Level"].value_counts()

HIGH            728
INTERMEDIARY    581
LOW             220
Name: CVD Risk Level, dtype: int64

classes are imbalanced.... maybe will need to add rows